# Cleopatra palettes on **real** GIS data

This notebook shows every palette in `cleopatra.palettes` applied to real geospatial rasters.
The data is read with **[pyramids](https://github.com/Serapieum-of-alex/pyramids)** (a GDAL-backed GIS
library that plots through cleopatra), so nothing here is synthetic:

- a real **digital elevation model** (Rhine basin, 5 km) for the *sequential* and *diverging* palettes, and
- a real **Sentinel-2 land-cover class** raster for the *qualitative* palettes.


In [ ]:
%matplotlib inline
# Use the in-development cleopatra (feature branch) if this is a dev checkout,
# otherwise the installed one. pyramids supplies the GDAL data reader.
import os, sys
_wt = r"C:/python-environments/worktrees/cleopatra/perceptual-palettes/src"
if os.path.isdir(_wt) and _wt not in sys.path:
    sys.path.insert(0, _wt)

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from pyramids.dataset import Dataset

import cleopatra
from cleopatra.palettes import available_palettes, get_palette, preview_palettes
print("cleopatra:", cleopatra.__version__, "|", len(available_palettes()), "palettes")

DATA = Path("../data/gis")   # kernel CWD is the notebook's own folder

def read_raster(path, band=0):
    """Read a raster into a (2D array, [xmin,xmax,ymin,ymax]) with nodata -> NaN."""
    ds = Dataset.read_file(str(path))
    arr = np.asarray(ds.read_array(), dtype=float)
    if arr.ndim == 3:
        arr = arr[band]
    nod = ds.no_data_value[0] if ds.no_data_value else None
    if nod is not None and np.isfinite(nod):
        arr = np.where(np.isclose(arr, nod), np.nan, arr)
    r, c = arr.shape
    x0, dx, _, y0, _, dy = ds.geotransform
    return arr, [x0, x0 + c * dx, y0 + r * dy, y0]

dem, dem_ext = read_raster(DATA / "DEM5km_Rhine_burned_fill.tif")
classes, cls_ext = read_raster(DATA / "sentinel-classes.tif")
print("DEM:", dem.shape, "elevation range", (np.nanmin(dem).round(1), np.nanmax(dem).round(1)), "m")
print("land-cover classes present:", np.unique(classes[np.isfinite(classes)]).astype(int))

## The palette registry at a glance

`preview_palettes()` renders the whole registry as a grouped swatch grid — continuous kinds as smooth ramps,
`qualitative` as discrete swatches.

In [ ]:
fig = preview_palettes()
fig.set_size_inches(9, 7)
plt.show()

## Sequential & diverging palettes on a real DEM

Each continuous palette is paired with the matplotlib norm its `kind` implies (`Palette.default_norm`):
a plain `Normalize` for sequential ramps, a symmetric `CenteredNorm` for diverging ones. The field is the
Rhine-basin elevation model.

In [ ]:
cont = [n for n in available_palettes()
        if get_palette(n).kind.value in ("sequential", "diverging")]
ncol = 3
nrow = int(np.ceil(len(cont) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(13, 3.1 * nrow))
for ax, name in zip(axes.ravel(), cont):
    p = get_palette(name)
    ax.imshow(dem, extent=dem_ext, origin="upper",
              cmap=p.to_colormap(), norm=p.default_norm(dem))
    ax.set_title(f"{name}  ({p.kind.value})", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
for ax in axes.ravel()[len(cont):]:
    ax.set_visible(False)
fig.suptitle("Rhine DEM (5 km) — sequential & diverging palettes", fontweight="bold")
fig.tight_layout()
plt.show()

## Qualitative palettes on real land-cover classes

The Sentinel-2 raster carries a handful of discrete land-cover classes. A `qualitative` palette maps each
class code to one swatch (a `ListedColormap`), so the classes stay crisp and maximally distinguishable.

In [ ]:
from matplotlib.colors import ListedColormap, BoundaryNorm

codes = np.unique(classes[np.isfinite(classes)]).astype(int)
k = len(codes)
qual = [n for n in available_palettes("qualitative")]
fig, axes = plt.subplots(1, len(qual), figsize=(6.2 * len(qual), 5.2))
for ax, name in zip(np.atleast_1d(axes), qual):
    swatches = list(get_palette(name).colors)[:k]
    cmap = ListedColormap(swatches)
    norm = BoundaryNorm(np.arange(k + 1) - 0.5, k)
    # remap arbitrary class codes to 0..k-1 for display
    remap = {c: i for i, c in enumerate(codes)}
    idx = np.vectorize(lambda v: remap.get(v, np.nan))(classes)
    im = ax.imshow(idx, extent=cls_ext, origin="upper", cmap=cmap, norm=norm)
    ax.set_title(f"Sentinel-2 land cover — {name}", fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
    cb = fig.colorbar(im, ax=ax, ticks=range(k), shrink=0.7)
    cb.ax.set_yticklabels([f"class {c}" for c in codes])
fig.tight_layout()
plt.show()